## Weekly shift handover (live MongoDB)

Loads shift reports from the CastNet **`reports`** collection for a date range, then summarises **only equipment whose name starts with `DCM`** (e.g. `DCM 1`, `DCM 10`), in **numerical machine order** (1, 2, … 10, not lexicographic). Other assets (Heller, machine shop, etc.) are skipped. Configure MongoDB via env vars (`CAST_LLM_MONGODB_URI`, etc.) or defaults in [settings.py](../src/cast_llm/settings.py).

In [1]:
import json
import re
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

from IPython.display import Markdown, display
from ollama import chat

from cast_llm import fetch_reports_date_range, get_project_root, group_by_equipment

### Date range (UK plant; Mongo `date_iso` is UTC)

Pick the wall-clock window in `Europe/London`, then convert to UTC for the query. Adjust the example datetimes to the week you need (e.g. the week before a Sunday 06:00 handover).

In [2]:
UK = ZoneInfo("Europe/London")
UTC = timezone.utc

# Example: one calendar week in UK local time
start_local = datetime(2026, 3, 29, 0, 0, 0, tzinfo=UK)
end_local = datetime(2026, 4, 4, 23, 59, 59, tzinfo=UK)

start_utc = start_local.astimezone(UTC)
end_utc = end_local.astimezone(UTC)
print("Query UTC range:", start_utc, "→", end_utc)

Query UTC range: 2026-03-29 00:00:00+00:00 → 2026-04-04 22:59:59+00:00


### Fetch from MongoDB and group by DCM

In [3]:
def _is_dcm_equipment(name: str) -> bool:
    """CastNet uses labels like 'DCM 1'; only those are included in the handover."""
    return name.strip().upper().startswith("DCM")


def _dcm_numeric_sort_key(name: str) -> tuple[int, str]:
    """Order DCM 1, DCM 2, … DCM 10 (not lexicographic DCM 1, DCM 10, DCM 2)."""
    m = re.search(r"DCM\s*(\d+)", name.strip(), flags=re.IGNORECASE)
    if m:
        return (int(m.group(1)), name)
    return (10**9, name)


rows = fetch_reports_date_range(start_utc, end_utc)
by_dcm = group_by_equipment(rows)
dcm_equipment_order = sorted(
    (k for k in by_dcm if _is_dcm_equipment(k)),
    key=_dcm_numeric_sort_key,
)

print(f"Total reports in range: {len(rows)}")
print(f"DCM-only equipment keys (numerical order): {len(dcm_equipment_order)}")
for k in dcm_equipment_order:
    print(f"  {k}: {len(by_dcm[k])} report(s)")

Total reports: 520  |  DCM keys: 28
  DCM 1: 54 report(s)
  DCM 10: 49 report(s)
  DCM 12: 43 report(s)
  DCM 15: 33 report(s)
  DCM 16: 55 report(s)
  DCM 2: 43 report(s)
  DCM 3: 41 report(s)
  DCM 5: 3 report(s)
  DCM 6: 40 report(s)
  DCM 7: 54 report(s)
  DCM 9: 58 report(s)
  General Comments: 19 report(s)
  Haas Lathe: 1 report(s)
  Heller 1: 1 report(s)
  Heller 10: 1 report(s)
  Heller 11: 2 report(s)
  Heller 13: 1 report(s)
  Heller 14: 2 report(s)
  Heller 3: 2 report(s)
  Heller 4: 2 report(s)
  MS CMM: 2 report(s)
  MS Conveyor: 1 report(s)
  MT82 Wash: 1 report(s)
  Machine Shop: 1 report(s)
  Mecwash: 2 report(s)
  No 1 Gauge Station: 1 report(s)
  Rosler: 1 report(s)
  (no equipment): 7 report(s)


### System prompt

In [4]:
with open(
    f"{get_project_root()}/prompts/weekly_handover_system.md", encoding="utf-8"
) as f:
    system_prompt = f.read()

### Run the model (one call per DCM)

Change `OLLAMA_MODEL` to match a model you have pulled locally.

In [5]:
OLLAMA_MODEL = "gpt-oss:20b"

sections: list[str] = []
sections.append(
    f"# Weekly handover summary\n\n"
    f"**UTC range:** {start_utc.isoformat()} – {end_utc.isoformat()}  \n"
    f"**UK window:** {start_local.date()} – {end_local.date()}\n"
)

for equipment in dcm_equipment_order:
    reports = by_dcm[equipment]
    if not reports:
        continue
    label = equipment
    user_content = (
        f"## Task\n"
        f"Summarise for **Sunday morning startup** on this machine only. "
        f"Be brief (see system prompt limits). "
        f"Weight **end-of-week** and **ongoing or linked** issues; "
        f"skip or minimise early-week problems that were fixed and did not recur.\n\n"
        f"## Range (UTC)\n{start_utc.isoformat()} – {end_utc.isoformat()}\n"
        f"(Mongo dates are UTC; plant operates UK time. Use report dates for recency.)\n\n"
        f"## equipment\n{json.dumps(label)}\n\n"
        f"## Shift reports (JSON array)\n{json.dumps(reports, indent=2)}"
    )
    stream = chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
        ],
        stream=True,
        keep_alive=-1,
        options={"num_ctx": 128000},
    )
    chunks: list[str] = []
    for chunk in stream:
        chunks.append(chunk["message"]["content"])
    body = "".join(chunks)
    sections.append(f"## {label}\n\n{body}\n")
    print(f"Done: {label} ({len(reports)} reports)")

full_response = "\n---\n\n".join(sections)

Done: DCM 1 (54 reports)
Done: DCM 10 (49 reports)
Done: DCM 12 (43 reports)
Done: DCM 15 (33 reports)
Done: DCM 16 (55 reports)
Done: DCM 2 (43 reports)
Done: DCM 3 (41 reports)
Done: DCM 5 (3 reports)
Done: DCM 6 (40 reports)
Done: DCM 7 (54 reports)
Done: DCM 9 (58 reports)
Done: General Comments (19 reports)
Done: Haas Lathe (1 reports)
Done: Heller 1 (1 reports)
Done: Heller 10 (1 reports)
Done: Heller 11 (2 reports)
Done: Heller 13 (1 reports)
Done: Heller 14 (2 reports)
Done: Heller 3 (2 reports)
Done: Heller 4 (2 reports)
Done: MS CMM (2 reports)
Done: MS Conveyor (1 reports)
Done: MT82 Wash (1 reports)
Done: Machine Shop (1 reports)
Done: Mecwash (2 reports)
Done: No 1 Gauge Station (1 reports)
Done: Rosler (1 reports)
Done: (no equipment set) (7 reports)


### Preview

In [6]:
display(Markdown(full_response))

# Weekly handover summary

**UTC range:** 2026-03-29T00:00:00+00:00 – 2026-04-04T22:59:59+00:00  
**UK window:** 2026-03-29 – 2026-04-04

---

## DCM 1

During the week DCM 1 struggled to maintain a steady run.  The die repeatedly suffered biscuit‑runner breakage when the bottom core pulled, prompting a series of spray, water‑heater and core‑temperature adjustments that never fully resolved the problem.  Porosity and bending remained the most common part‑quality failures, with fast‑track passes only after several spray‑timing changes or temporary tool‑room fixes.  A succession of oil leaks (tie‑bar, shot‑raise, F/H and side‑core plugs) forced frequent maintenance interventions, while the datum N on the die stayed out of tolerance despite repeated polishing and replacement.  Production was intermittently halted, with many shifts reporting little to no castings, yet a few quick‑runs did achieve 6/6 fast‑track results after spray‑timing tweaks.

**Key items**

- **Biscuit core / runner**: repeated breaking when bottom core pulled; many attempts to raise water heater, adjust sprays and lower core temperature, still unresolved.  
- **Porosity & bending**: consistent fast‑track failures; porosity in seal face/bore; bending noted on several shifts.  
- **Datum N**: out of tolerance repeatedly; tool‑room polished, replaced pins, F/H off, but still non‑concentric.  
- **Oil leaks**: tie‑bar cylinder, shot‑raise pipe, F/H, side‑core plugs and spray head leaks; each repaired by maintenance.  
- **Spray & heater adjustments**: timers increased to 4.5‑5.5 s, new spray position added, pressure raised to 60 psi, water heater cycled between 70‑90 °C.  
- **Maintenance actions**: tightened SQZ pin, refitted burst pipe, replaced impression plugs, spray head repairs, fan cleaning.  
- **Production**: many shifts stopped or ran very low; occasional 6/6 fast‑track after adjustments, otherwise poor throughput.  
- **Fast‑track status**: passes only after spray‑timing or tool‑room fixes; otherwise 0‑5/6 or porosity‑related fails.

---

## DCM 10

**DCM 10 – KM RDM Carrier ‑ W‑309 ‑ 2.2 (29 Mar – 4 Apr, UK time)**  
This machine spent most of the week in a state of repeated stoppages, largely driven by core‑pin failures and a persistent water‑leak in the shot tip. Operators and maintenance teams performed frequent de‑solders, rock‑linising and vacuum clearances, but the core‑pin cracks and internal tearing kept recurring. Insert‑load and extract‑robot faults added further cycle‑time loss, while a ladle‑probe fault and a safety‑edge strip failure added to the maintenance burden. Despite the interventions, production could only recover intermittently, with a noticeable backlog of de‑solder‑and‑clean‑runs each shift.

**Key items to note**

- **Core pin:** broken behind top core on 29 Mar, 30 Mar, 31 Mar, 1 Apr, 2 Apr, 3 Apr – repeatedly repaired or replaced.  
- **Water leak:** shot‑tip leak on 29 Mar (fixed) and a second leak reported on 31 Mar (re‑soldered).  
- **Ladle probe:** faulted on 29 Mar, 30 Mar, 31 Mar, 1 Apr, 3 Apr – rewired or re‑connected each time.  
- **Vacuum:** blocked on 29 Mar, 31 Mar – cleared twice; a vacuum pipe full of water noted on 3 Apr.  
- **Extract / insert load:** extract‑robot failed to load pucks on 29 Mar, 31 Mar, 3 Apr; insert load tweaked repeatedly on 31 Mar, 1 Apr.  
- **Tear / cracking:** internal lightening pocket tearing and die wall cracking reported from 29 Mar to 3 Apr – de‑soldered and monitored.  
- **De‑solder / rock‑linise:** 40+ de‑solder operations and rock‑linising of top core throughout the week.  
- **Maintenance:** small valve change (open/close) on 3 Apr; safety‑edge strip reset on 1 Apr; encoder magnet/translator replacement on 3 Apr.  
- **Other:** “M/H off” due to broken core pin on 30 Mar, 31 Mar, 1 Apr, 2 Apr; “M/H off” on 31 Mar due to poor flow after core pin removal.

---

## DCM 12

**DCM 12 – 29 Mar – 4 Apr 2026**  
The plant ran the Dana MLA High Rear Carrier – W‑490 – 1.3 throughout the week, with a brief W‑491 run on 4 Apr. Production was repeatedly halted by trim‑press trips, drill‑tip loss and broken switch heads – all of which were repaired by toolroom and maintenance. Hydraulic leaks (FH manifold, Q/R seals) and a low water level were fixed, while the Kuka robot suffered air‑line and sensor failures that caused dropped castings and an ethernet fault. Numerous de‑solder operations at shift ends signalled ongoing tooling instability, and cavity changes were carried out under pressure with vacuum‑cyclone interference and a lower‑foot water leak. The week ended with many repairs completed, but the trim press and robot systems still required stabilisation.

- **Trim press** – repeated trips from missing cutters and damaged drill switch heads; toolroom replaced cutters, maintenance replaced and reset switches.  
- **Drill** – tip lost, replaced; head switch broken, replaced; one‑way valve in lube tank replaced to stop overflow.  
- **Hydraulics** – FH manifold leak fixed with short hose; Q/R seals tightened; Q/T water level low, valve replaced.  
- **Kuka robot** – air‑line snagged/broken on several shifts, sensors missing, dropping castings; ethernet fault rebooted on 1 Apr; retaught on pick‑up failures.  
- **De‑solder** – dozens of de‑solder events at shift ends, mainly on W‑490, indicating tooling instability.  
- **Cavity change** – performed under vacuum‑cyclone interference; dietanic extended to avoid blockage; lower‑foot water leak forced FH off.  
- **Porosity & oil heater** – porosity found in pinion and feet; oil heater low temperature corrected on 2 Apr.  
- **Toolroom repairs** – breakout on M/H die welded; extra weld added to haldex striker arm; cavity change adjustments and puck deslugger clearing.

---

## DCM 15

**Overall picture**  
DCM 15 ran the Getrag B6 – W‑154 – 14 die through every shift of the week, but frequent interruptions kept the plant busy.  Biscuit build‑up, gripper‑sensor damage, a switched‑off water supply and several broken core or squeeze pins forced regular de‑solder and toolroom repairs.  Despite these hiccups, fast‑track checks were usually 6/6 and most CMM/gauge/x‑ray inspections passed, with only occasional porosity or staining that were dealt with by spray adjustments and process checks.  Maintenance and toolroom teams carried out core changes, poppet and thurner replacements, vacuum cleaning and conveyor resets while operators performed daily cleaning, de‑solder and die‑face checks.

**Concrete items**

- **Tooling – Core pin:** M/H top slide pin G29 broken; core changed, sheet completed.  
- **Tooling – Squeeze pin:** Pin on bore 66 damaged; new pins/sleeves installed, ejector & poppet repaired.  
- **Tooling – Poppet & vacuum:** Vacuum poppet broke, metal filled pipework; new poppet and thurner block fitted, vacuum cleaned.  
- **Process – Water supply:** Water off caused no spray; issue resolved after restart.  
- **Process – Biscuit build‑up:** Biscuits damaged gripper sensor; sensor replaced and operator cleared buildup.  
- **Quality – Porosity:** Bore 59, 65, 66 porosity noted; spray adjustments and extra checks performed.  
- **Quality – Staining:** Top and bore 75 staining; spray adjustments and die‑face cleaning carried out.  
- **Maintenance – Scrap & quench:** Scrap conveyor reset; quench tank debris cleared and vacuum cleaned.  
- **Operator actions:** De‑solder performed each shift; die‑face and Q/T debris cleared; multiple fast‑track checks (mostly 6/6) recorded.

---

## DCM 16

**DCM 16 – KM RDM Carrier – W‑308 – 1.1**  
Over the week the unit ran the same die but was frequently stopped by a chain of mechanical and tooling faults. Production was interrupted by repeated insert‑conveyor failures, quench‑jig looseness and misloads at the trim, which required several maintenance actions before the machine could get back to steady runs. The quality checks were largely green, with only occasional trim‑damage and indented‑sleeve defects that were cleaned or de‑soldered. Toolroom and maintenance kept the machine running by replacing core‑pins, tightening bolts, swapping sensors, welding supports and re‑tuning the vent‑knock and quench‑jig positions. The day‑to‑day handover involved a steady stream of minor fixes and a few days of planned downtime for larger repairs and fabrication work on the cell.

**Key concrete items**

- **Die / Tool:** KM RDM Carrier – W‑308 – 1.1 was the only die run; several die‑tool adjustments (vent‑knock positions, quench‑jig bolts, core‑pin replacements) were made.  
- **Production:** Frequent stoppages due to insert‑conveyor miss‑loads, quench‑jig loose bolts, trim‑misloads and KUKA errors; machine eventually ran 1 h 20 min with no stops on 29 Mar.  
- **Quality:** Indented sleeve G‑face and A‑face OK; trim damage on A‑face on 29 Mar, X‑ray/gauge CMM all within spec; occasional de‑solder required.  
- **Maintenance:** Replaced insert‑conveyor sensor, welded knock‑off jig support, fixed broken air fitting on KUKA gripper, tightened quench‑tank fixings, repaired bent support bar on vent‑knock.  
- **Toolroom:** Removed washers under knock‑off cylinder, fitted new coil, replaced Haldex core‑pin, fitted puck deslugger spacer plate, replaced squeeze pins, cleaned return light switch on trim puck catcher.  
- **Management:** Re‑taught KO jig moves, added machined plate under knock‑off cylinder, re‑ordered lock‑out clasp for trim press, updated robot logic to check part‑removed flag.  
- **Shift handover:** Several days of fabrication work on the cell (Mar 30 – Apr 2) required the machine to be stopped for 30‑60 min periods, but each shift was handed off with the die and tooling status clearly noted.  
- **Recurring issue:** The quench jig remains a weak point – loose bolts and misloads were recorded on 29 Mar, 31 Mar and 1 Apr, with repeated tightening actions needed.  

Overall, DCM 16 stayed productive but was heavily dependent on maintenance fixes and quality checks to keep the die running within spec.

---

## DCM 2

**DCM 2 – 29 Mar → 4 Apr 2026**  
The line spent most of the week battling the extractor robot – the gripper would repeatedly fail to open or pick up a casting, causing a string of minor stoppages (most notably the 22:00‑02:50 shift‑3 outage). Production kept moving, but with many short‑stop events and several castings held for quality checks. Die‑opening problems were largely cured by greasing tie bars and cleaning sensors, though a die‑flash/ core‑welding incident on 1 Apr required a tool‑room fix. Porosity was the main quality issue, with haldex voids noted in the first three days of the period. Maintenance was busy cleaning sensors, swapping jammed conveyor carriages, patching a spray‑robot hose leak, repairing a damaged power cable, and repeatedly updating the extraction programme.

**Key items**

- **Extract robot gripper** – sensor faults and open‑close loops; replaced faulty sensor (Kuka), cleaned switches, but still a loop on the 4 Apr shift‑3.  
- **Die opening** – tie‑bar greasing and sensor cleaning restored full opening; die flashing and core welding required a tool‑room repair on 1 Apr.  
- **Insert conveyor** – multiple carriage jams; sensors reset and new carriage set installed during PM.  
- **Porosity** – haldex face porosity noted 29‑31 Mar; X‑ray/gauge/CM checks passed after remedial re‑soldering.  
- **Maintenance interventions** – spray‑robot hose leak fixed (1 Apr), air‑leak on spray pipe repaired (1 Apr), damaged top‑box power cable replaced (2 Apr), quench‑tank sensor swapped (2 Apr).  
- **Program updates** – extra line added to extract routine (2 Apr), wait‑times inserted, and DCM 1st Cycle flag removed to stop mid‑run stalls.  
- **Production stoppages** – most downtime from 22:00‑02:50 shift‑3 (extract failure), 18:49‑20:00 shift‑3 (insert carriage jam), and 1250‑on‑shift on 2 Apr when DCM 1 was backup.  
- **Quality checks** – routine X‑ray, gauge, CMM all OK except the noted porosity; de‑soldering performed on several days.  
- **PM and tool‑room work** – core weld repair, stamp removal, and inspection done during 1‑4 Apr.  
- **Operator actions** – multiple re‑loads of the spray‑robot programme, manual clearing of metal build‑up, and de‑soldering to keep the line running.

---

## DCM 3

The RDU Cover W‑356 2.9/2.10 die ran on DCM 3 throughout the week, but production was repeatedly interrupted by hydraulic leaks, squeeze‑pin failures and dead‑bar switch faults. The machine was down for 3–4 hours on 30 March after sensors and the dead‑bar switch had to be re‑aligned, and several shorter stoppages occurred on 29 March when squeeze pins failed or hydraulic leaks appeared. Quality checks flagged porosity in the main bore and ear, necessitating repeated desoldering, although visual, CMM and X‑ray checks generally passed once the die was cleaned. Tooling work included replacing a worn nib, repairing die‑clamp cylinders, and cleaning shot‑rod seals, while process tweaks – correcting Kuka loading, adjusting biscuit size and shot‑bead dosage – were made to reduce breakage and staining.

**Key events**

- **Die / tool** – Squeeze‑pin failure on 29 March; nib replaced 31 March; die‑clamp cylinder hydraulic leaks repaired 1 April and 2 April.  
- **Maintenance** – Dead‑bar limit switch cleaned 29 March and 30 March; mis‑load sensors realigned 1 April; trim sensors replaced 2 April; shot‑rod seal and hydraulic o‑rings replaced 31 March and 2 April.  
- **Quality** – Porosity in main bore/ear recorded 30 March and 31 March; X‑ray of cavity 2.9 poor but improved 2 April; de‑solder of 2.9 slot, TR, bushing bore performed.  
- **Process** – Kuka mis‑loading due to 4 extra program lines on 30 March; biscuit breakage prompted change of biscuit size and shot‑bead dosage 1 April; pour‑position adjustment to reduce staining 3 April.  
- **Downtime** – Machine down 3–4 h on 30 March for sensor realignment and dead‑bar switch cleaning; several short shutdowns (Black 29 March, Blue 29 March) due to squeeze‑pin failures and hydraulic leaks.

---

## DCM 5

Over the week DCM 5 was mainly used for a squeeze‑pin trial on the Hanon Centre Housing – W‑723 – A1/A2/A3/A4 die. Trial parts were produced on the bench and pallet, with squeeze‑pin times ranging from 2 to 10 s and the timing marked on the castings. On 2 April the furnace loader motor was replaced – the old motor stripped and a new one wired in – and the change was completed during the Green shift. No downtime or quality issues were recorded in the reports, indicating a smooth operation for the period.

- **Trial run** – Hanon Centre Housing – W‑723 – A1/A2/A3/A4 die – squeeze‑pin trial parts made on bench/pallet.  
- **Timing** – squeeze‑pin cycle 2–10 s, marked on castings.  
- **Maintenance** – new furnace loader motor installed on 2 April.  
- **Maintenance** – motor stripped, re‑wired, old motor removed (Kelvin).  
- **Shift** – Blue shift (trial) and Green shift (maintenance) logged.  
- **No downtime** – plant ran continuously; no outages reported.  
- **No quality issues** – no rejects or defects mentioned.

---

## DCM 6

DCM 6 ran the Dana MLA High Front Carrier – W‑461 – 2.2 continuously throughout the week, but production was repeatedly interrupted by pin‑ and core‑pin failures, spray‑related sinking and hydraulic oil leaks that required frequent maintenance. Quality teams carried out a string of de‑solder/rock‑linising jobs, CMM/X‑ray checks and gauge‑re‑checks that largely restored acceptable part quality after each repair. Toolroom repeatedly replaced squeeze pins, core pins (V10, V11, V15, V17) and a broken trim punch, and addressed quench‑tank and shot‑rod leaks, creating a pattern of short shutdowns but steady recovery.

**Key items**

- **Tooling / die:** Dana MLA High Front Carrier – W‑461 – 2.2 was repeatedly de‑soldered and rock‑linised; squeeze pins, core pins (V10, V11, V15, V17) and a trim punch were replaced on several days.  
- **Hydraulic leak:** Large oil leak on the M/H squeeze‑pin pipe repaired 2026‑04‑03; earlier shot‑rod leak checked 2026‑03‑31; valve leak addressed 2026‑03‑29.  
- **Spray / sink issues:** Pin‑holes in the lightning pocket and sinking on castings reported 2026‑03‑30–31; spray checks performed and adjusted but the problem recurred.  
- **Quality checks:** Repeated CMM, X‑ray and gauge checks were mostly OK; gauge failures on bottom/top boss fixed by de‑solder; porosity noted 2026‑03‑31 and 2026‑04‑02, resolved after de‑solder.  
- **Trim punch:** Broken on 2026‑04‑01, replaced 2026‑04‑01; again bent on 2026‑04‑03, cleared debris and replaced.  
- **Maintenance actions:** Trim‑press sensor adjusted 2026‑03‑31; quench tank temperature restored 2026‑04‑01; shot‑rod O‑ring replaced 2026‑04‑02.  
- **Downtime / recovery:** Shift start delayed 08:30 on 2026‑03‑29 due to valve leak; M/H off for squeeze‑pin replacement on 2026‑03‑31; multiple short stoppages for de‑solder, core‑pin replacement and hydraulic pipe repair.  
- **Repeat issues:** Recurring pin‑hole/sink problems in the lightning pocket; core‑pin looseness/unwinding; hydraulic pipe leaks.

---

## DCM 7

**Overall picture (Mar 29 – Apr 4)**  
DCM 7 ran the KM PTU Carrier – W‑306 – 2.1 die for the whole week, but production was repeatedly stalled by porosity in the pinion and M8 areas and a succession of die‑tool and hydraulic faults. Multiple maintenance actions were carried out – shot‑seal, dock‑seal, oil‑seal, core‑insert and saw‑blade changes – and the cycle time was briefly cut to ~109–110 s after spray‑timing tweaks, yet quality remained inconsistent and downtime stayed high. The handover notes repeatedly flagged trim damage, core‑tip wear and persistent porosity, signalling that further process optimisation is required.  

**Concrete items**

- **Die:** KM PTU Carrier – W‑306 – 2.1 ran throughout; several rebuilds of the shot end and cleaning of the sleeve sprayer.  
- **Porosity:** Repeated porosity in pinion and M8; operators logged 5/6 or 6/6 part quality; spray‑timing adjustments were tried but results were inconsistent.  
- **Maintenance – Shot seal:** Shot‑rod gland seal leaked heavily (Blue shift) and main shot seal was replaced after the Saturday oil leak.  
- **Dock & Trim:** Dock seals and dock switch were replaced; trim damage was recorded repeatedly; saw blade replaced and a low‑level alarm fitted to the saw‑lube tank.  
- **Hydraulic leak:** Trim‑press top O‑ring leak found and replaced; hydraulic leak from valve in trim press was addressed.  
- **Core issues:** Core‑insert speed/pressure was adjusted; bottom‑slide core plug replaced after robot mis‑load; core tip was replaced twice due to wear and porosity.  
- **Cycle time:** Cycle time briefly reduced to ~109‑110 s after spray dwell and timer adjustments; earlier cycles were 121 s.  
- **Quality handover:** Quality checks flagged porosity on multiple parts; operators were instructed to gauge 100 % and check for graphite; X‑ray quality remained poor for many shifts.  
- **Tooling & Alignment:** Shot alignment checks were performed after the hydraulic leak; dock alignment completed; core pressure and encoder were inspected.  
- **Handover notes:** Several reports noted “hand‑ing over” with ongoing trim damage, core‑tip changes and pending saw‑blade adjustments.

---

## DCM 9

DCM 9 spent most of the week being tuned and repaired rather than producing. The trim‑saw motor and its oil seals repeatedly failed, requiring motor replacements, seal swaps and speed adjustments. The scrap conveyor jammed on several shifts because of bent flights or oil leaks, and shot‑docking faults forced operators to pull maintenance in each shift. Die‑related porosity and blisters, especially in the LH 1.2 and RH 1.2 cavities, were a recurring issue that was dealt with by cleaning the die, moving a forward‑sticking pin, and adjusting timers. Quality checks consistently highlighted poor fill and shiny parts, but most runs were salvaged after die and process adjustments.

**Key items**

- **Trim saw motor / oil seals:** Repeated failures; motors and oil seals were replaced and speed set on several occasions during the week.  
- **Scrap conveyor:** Jammed due to bent flights and oil leaks; fixed by straightening flights, tightening guards and replacing leak fittings.  
- **Shot docking faults:** Dock conditions not met for Sora and other faults; switches replaced, dock clamps tightened and alignment checked.  
- **Die porosity / blisters:** LH 1.2 and RH 1.2 cavities showed porosity and blisters; die pin was 5 mm forward, die pulled off, and cleaning/ timer adjustments restored quality.  
- **Graphite system:** Nozzle repeatedly blocked and noisy; graphite pump replaced, graphite tank removed and stored in the tool room.  
- **Safety latch:** Shot safety latch stuck out; cylinder removed, cleaned and greased.  
- **Quality checks:** Poor fill and shiny parts were common, but x‑rays were mostly OK after process adjustments.  
- **Maintenance interventions:** Thermocouple replacement, graphite‑system motor replacement, saw blade replacement, door cable tie, and other quick fixes were performed throughout the week.  
- **Conveyor & e‑stop:** E‑stop failed to reset due to gate scrap chute interference; resolved by tying a metal cable to secure the gate.

---

## General Comments

The week’s “General Comments” entries show a routine blend of preventive maintenance and toolroom preparation work. Toolroom staff completed trim‑tool setup and storage tasks for Dana and km‑ptu tools, worked on a 350 CVR M/H cavity, and finished datestamping. Maintenance performed scheduled PMs on a wide range of ladders and knock‑out units, replaced a DMC fire‑wall traverse cylinder, fixed RDU‑cover clamps, and carried out electrical certifications and weekly fire checks. Minor repairs such as a linishing gun were also finished, and the ODYSSEY line remained booked off for the period, with no major incidents reported.

**Toolroom**: Trim‑tool prep – Dana rear carrier, Dana HRC, km ptu; RDU cover refit and cavity assembly; datestamps completed.  
**Toolroom**: 350 CVR M/H cavity work continued toward go‑in.  
**Maintenance**: Metal sticks with magnets fabricated for DCM 10, 12, 16.  
**Maintenance**: DMC fire‑wall – traverse cylinder replaced; RDU‑cover clamps corrected and verified.  
**Maintenance**: Electrical certifications, weekly fire checks, and linishing‑gun repair completed.  
**Maintenance**: Weekly ladder PMs on DCM 01–15 and Step Ladder 03; knock‑out unit PMs on K01–K07 with minor adjustments.  
**Maintenance**: Energy Centre weekly PM performed.  
**Booking**: ODYSSEY booked off for the entire week.

---

## Haas Lathe

The Haas Lathe had a brief outage at the start of the week caused by a Kuka robot fault. The operator re‑homed the robot, released the held part, started the lathe again and has been running it normally since. No other incidents or maintenance actions were recorded during the period.

**Key points**

- **Issue:** Kuka robot fault caused downtime at the start of the shift.  
- **Action:** Robot was homed, part released, lathe restarted.  
- **Status:** Lathe running OK currently.  
- **Repeat:** No repeat events noted.  
- **Maintenance:** None required or performed.

---

## Heller 1

Heller 1 had a brief period of downtime during the week, mainly due to a fault on its B‑axis. The maintenance team had to lift and clean the floor before any repairs could start, delaying the shift. No other incidents or significant changes were reported for the plant during this period.  

**Key points**  
- **Axis:** B‑axis fault halted production.  
- **Maintenance:** Floor lifted and cleaned before work could resume.  
- **Downtime:** Brief, caused by the B‑axis issue and floor preparation.

---

## Heller 10

Over the week Heller 10 had a single recorded maintenance event. The encoder on the Tool Breakage arm was suspected to be faulty, was replaced, and tested for 30 minutes with no issues reported. No other incidents, downtime, or production changes appear in the supplied reports, so the line was running normally for the rest of the period.

- **Maintenance / Tooling:** Replaced and tested the encoder on the Tool Breakage arm, confirmed OK after 30 minutes.  
- **Production:** No downtime or production problems reported.

---

## Heller 11

During the week the Heller 11 ran on the Black shift but faced two consecutive tap‑break incidents in its Hofer Gearbox. The M5 tap (HG167) fractured during the first cycle of production and had to be removed, with a stop inserted into the program to step through the drill and tap. The following day the same tap broke again, with no spare available, prompting an email to Carly. The GI224 component was taken from the Gemini Intermediate rack and placed on the setter desk, and the parts produced before the break were inspected at the M/C and marked for CMM measurement.

- **Tool failure:** M5 tap (HG167) broke twice; no spares on hand.  
- **Action taken:** Tool removed; program stop inserted to step through drill/tap.  
- **Tool relocation:** GI224 removed from Gemini Intermediate, tagged, placed on setter desk.  
- **Communication:** Email sent to Carly for spare parts.  
- **Quality check:** Parts at M/C appear OK; castings marked for CMM.  
- **Downtime:** Caused by tap break in both cycles.

---

## Heller 13

During the week the Heller 13 operated the MT82 die. On the Black shift the operator noted that holes 1 and 2 were too tight. The operator replaced T1004 and re‑gauged the parts at the machine, finding all dimensions within specification. No other incidents were reported, so the shift proceeded without noticeable downtime.  

- **Die / tool:** MT82 – holes 1+2 were tight.  
- **Maintenance:** T1004 replaced.  
- **Quality:** Parts gauged at machine; all OK.  
- **Production:** No downtime reported; shift ran normally.

---

## Heller 14

**Overall picture (29 Mar – 4 Apr 2026)**  
During the week Heller 14 operated normally apart from two brief interruptions. On 30 March a monitoring fault in the Tesla system (monitoring‑time‑reset treatment channel 1 exceeded) forced operators to shut the motors off to reset the alarm. On 1 April a coolant leak re‑emerged; operators cleared swarf‑filled coolant channels, pulled back the Z‑axis concertinas and removed the access pallet to clean the bed. No production figures or quality defects were logged, and the machine was brought back online after the cleaning.

**Concrete items**  
- **Subsystem:** Tesla monitoring fault – motors shut down to reset alarm (30 Mar).  
- **Fault:** Cooling‑channel leak – swarf blockage cleared; reoccurred later in the shift (1 Apr).  
- **Maintenance:** Z‑axis concertinas pulled back, pallet removed to clear bed coolant channels; reminder to adhere to monthly PM cleaning schedule.  
- **Recovery:** Machine restarted after manual cleaning; no further issues reported.

---

## Heller 3

During the week Heller 3 experienced two mid‑cycle stoppages.  The first was a weight‑relief‑pressure fault (700806 – B61.7) that halted the machine, and the second was a tool‑change door fault that prevented the door from fully closing.  In both cases the operator reset the machine and continued from the interruption point, with no additional downtime or quality impact noted.

**Key points**

- **Fault 1 – Weight relief pressure (700806, B61.7):** machine stopped mid‑cycle; reset and resumed.
- **Fault 2 – Tool‑change door not fully closed:** cycle halted; reset using individual functions, then resumed.
- **Recovery:** both incidents were fixed by operator reset; no die or tooling changes were required.
- **Quality:** no defects or quality issues were reported.

---

## Heller 4

Heller 4 had a quiet week of production with only two brief hiccups. The Black shift of 31 March saw an SBBK fault that was quickly reset with no tool damage, allowing the cycle to resume. The following Day shift (1 April) was interrupted by a hydraulic leak in the Getrag machining fixture – caused by a split pipe fitting and sheared mounting‑plate bolts – which was fixed by helicoiling the M8 fixing bolt hole and replacing the damaged pipework. No quality issues were reported and the machine returned to normal output after the maintenance.

**Key items**

- **SBBK fault (Mar 31)** – reset SBBK, tool OK, cycle continued.  
- **Hydraulic leak (Apr 1)** – split pipe fitting on Getrag fixture; sheared mounting‑plate bolts.  
- **Maintenance** – helicoiled M8 fixing bolt hole, replaced pipework/fittings.  
- **Production** – minimal downtime, output maintained.  
- **Quality** – no defects reported.

---

## MS CMM

Over the week MS CMM was largely in routine inspection mode. Probe checks on CMM1 were clean, while CMM2 flagged a fault on tool 10 – a tool that isn’t part of the current production run, so it didn’t cause any downtime. The team started a Tesla R/Hs measurement on CMM2, but the run was held pending fixture resolution. No machine outages were reported, but the tool‑10 fault and fixture reliability should be kept in mind for the next shift.

**Key points**

- **Probe checks:** CMM1 clear; CMM2 flagged a fault on tool 10.  
- **Tool:** Tool 10 fault noted, not used in current production.  
- **Run:** Tesla R/Hs measured on CMM2, held due to fixture issues.  
- **Fixture:** Pending resolution; hold until fixed.  
- **Maintenance:** No immediate actions required; tool 10 fault logged for review.  
- **Handover:** Craig Jones completed checks; next shift should verify fixture status before proceeding.

---

## MS Conveyor

**Overall picture:**  
During the week the MS Conveyor ran smoothly with no production‑level stoppages.  A sensor/switch after B6 pass had to be repositioned to keep parts moving; the button beside CNC 9 was acting sporadically, but the issue was resolved and the line has been running without further problems.  No quality or tooling incidents were reported.

**Key items**

- **Sensor / switch:** pulled back after B6 pass to keep parts flowing.  
- **Button (CNC 9):** intermittent operation, fixed – line back to normal.  
- **Status:** no further incidents, production continued uninterrupted.

---

## MT82 Wash

MT82 Wash only had a single interruption during the week.  
On 2 April the exit‑side estop jammed, taking the unit offline until a new estop and back‑box were fitted.  
After power was restored the controller flashed and did not show a temperature reading, but a quick tap brought it back online.  
No other downtime, quality or tooling issues were recorded for this DCM in the period.

**Key items**

- **Safety / Estop:** Exit‑side estop stuck – replaced with new estop and back‑box.  
- **Control panel:** After power‑on the controller flashed, no temperature display; a tap restored normal operation.

---

## Machine Shop

Over the week, the only activity logged for Machine Shop was a maintenance task on the B6 pressure‑test bench.  
A flexible air line and an isolation valve were fitted in the red shift on 2 April, and two new air hoses were made for the shop.  
No production, quality or downtime incidents were reported.  
The note suggests that the isolation line should eventually be moved to galvanised pipework for better isolation.  

- **Maintenance:** Flexible air line and isolation valve fitted to the B6 pressure‑test bench.  
- **Air supply:** Two new air hoses installed for the machine shop.  
- **Recommendation:** Relocate isolation to galvanised pipework.  
- **Shift:** Red shift, 2 April.  
- **Status:** No production downtime recorded.

---

## Mecwash

During the week Mecwash stayed operational, but a couple of maintenance issues cropped up. Filters were removed and replaced to probe a suspected pressure drop; the wash‑side stayed at 0.3‑0.34 bar and the rinse at 0.3‑0.5 bar, so the problem remained unresolved yet production kept running. A more serious fault was the oil‑skimmer unit, which was found to be damaged; it was temporarily re‑assembled and the oil‑skimmer function was disabled on the HMI because the CH hot‑water pump was seized. No further downtime was recorded, but the hot‑water pump fault still requires a permanent fix.

**Filters:** swapped but pressure unchanged (wash 0.3‑0.34 bar, rinse 0.3‑0.5 bar); production continued.  
**Oil skimmer unit:** damaged; re‑assembled to keep the machine running; HMI disabled.  
**Hot‑water pump (CH):** seized; the fault caused the oil‑skimmer issue; remains unresolved.  
**Maintenance actions:** re‑assembled oil skimmer; disabled oil skimmer on HMI.  
**Production status:** Mecwash running with oil‑skimmer disabled, no downtime.

---

## No 1 Gauge Station

No 1 Gauge Station had a single recorded activity during the period: a calibration of the MT82 gauge. No downtime, quality complaints or further maintenance actions were logged. The calibration was carried out following the standard Sunday start‑up procedure, indicating that the station is being kept in the expected condition. With no other incidents reported, the station appears to be running smoothly and ready for the next week’s production.

**Key points**

- **Calibration:** MT82 Gauge station calibrated as per Sunday Start‑up Procedure.  
- **Maintenance:** No additional maintenance logged.  
- **Operation:** No production or quality issues reported.  
- **Handover:** Not applicable – only one report available.

---

## Rosler

Rosler experienced a minor maintenance issue during the reporting week. The elevator shaft developed a leak halfway up, which was promptly patched with silicone by the maintenance team. No other incidents or production disruptions were recorded in the available reports.  

**Maintenance:** elevator shaft leak patched with silicone.

---

## (no equipment set)

There are no reports for the specified equipment during this period.


### Save to `notebooks/weekly_handover.md`

In [7]:
out_path = f"{get_project_root()}/notebooks/weekly_handover.md"
with open(out_path, "w", encoding="utf-8") as f:
    f.write(full_response)
print("Wrote", out_path)

Wrote /home/castalum/dev/cast-llm/notebooks/weekly_handover.md
